In [21]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("Iceberg_Projeto") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/app/data/iceberg/iceberg_warehouse") \
    .getOrCreate()

print("Spark com Iceberg iniciado")

Spark com Iceberg iniciado


In [22]:
spark.sql("CREATE DATABASE IF NOT EXISTS local.db_teste")
print("Database criada")

Database criada


In [23]:
spark.sql("SHOW DATABASES IN local").show()

+---------+
|namespace|
+---------+
| db_teste|
+---------+



In [24]:
data = [
    ("Engenharia", 1),
    ("Dados", 2),
    ("Big Data", 3)
]

df = spark.createDataFrame(data, ["palavra", "valor"])
df.show()

+----------+-----+
|   palavra|valor|
+----------+-----+
|Engenharia|    1|
|     Dados|    2|
|  Big Data|    3|
+----------+-----+



In [25]:
df.writeTo("local.db_teste.tabela_v1").createOrReplace()

print("Tabela Iceberg criada")
spark.table("local.db_teste.tabela_v1").show()

Tabela Iceberg criada
+----------+-----+
|   palavra|valor|
+----------+-----+
|Engenharia|    1|
|     Dados|    2|
|  Big Data|    3|
+----------+-----+



26/04/26 17:05:27 WARN HadoopTableOperations: Error reading version hint file /app/data/iceberg/iceberg_warehouse/db_teste/tabela_v1/metadata/version-hint.text
java.io.FileNotFoundException: File /app/data/iceberg/iceberg_warehouse/db_teste/tabela_v1/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:160)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:372)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:976)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:318)
	

In [26]:
spark.sql("""
INSERT INTO local.db_teste.tabela_v1
VALUES ('Spark', 4)
""")

print("INSERT feito")
spark.table("local.db_teste.tabela_v1").show()

INSERT feito
+----------+-----+
|   palavra|valor|
+----------+-----+
|     Spark|    4|
|Engenharia|    1|
|     Dados|    2|
|  Big Data|    3|
+----------+-----+



In [27]:
spark.sql("""
UPDATE local.db_teste.tabela_v1
SET valor = 99
WHERE palavra = 'Dados'
""")

print("UPDATE feito")
spark.table("local.db_teste.tabela_v1").show()

UPDATE feito
+----------+-----+
|   palavra|valor|
+----------+-----+
|Engenharia|    1|
|     Dados|   99|
|  Big Data|    3|
|     Spark|    4|
+----------+-----+



In [28]:
spark.sql("""
DELETE FROM local.db_teste.tabela_v1
WHERE palavra = 'Engenharia'
""")

print("DELETE feito")
spark.table("local.db_teste.tabela_v1").show()

DELETE feito
+--------+-----+
| palavra|valor|
+--------+-----+
|Big Data|    3|
|   Spark|    4|
|   Dados|   99|
+--------+-----+



In [29]:
spark.sql("""
SELECT * FROM local.db_teste.tabela_v1.snapshots
""").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2026-04-26 17:05:...|5703929043029109960|               NULL|   append|/app/data/iceberg...|{spark.app.id -> ...|
|2026-04-26 17:05:...| 245569009277018274|5703929043029109960|   append|/app/data/iceberg...|{spark.app.id -> ...|
|2026-04-26 17:05:...|6968415655614766148| 245569009277018274|overwrite|/app/data/iceberg...|{spark.app.id -> ...|
|2026-04-26 17:05:...|5549337958564487626|6968415655614766148|   delete|/app/data/iceberg...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+



In [30]:
spark.sql("DESCRIBE TABLE local.db_teste.tabela_v1").show()

+--------+---------+-------+
|col_name|data_type|comment|
+--------+---------+-------+
| palavra|   string|   NULL|
|   valor|   bigint|   NULL|
+--------+---------+-------+

